# First-time insider buys: insiders buying for the first time

Find open-market purchases by insiders with no earlier purchase on record, by checking each buyer's Form 4 history by CIK.

Read it with charts at https://www.quantisnow.com/sec-api/guides/first-time-insider-buys. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A director who buys every quarter tells you little each time. One who has never bought before, and suddenly puts money in, tells you more. The API can't express "first" in one query, but two calls do it: pull this week's buys, then check each buyer's history by CIK.

## Start from this week's purchases

### Which insiders bought more than $1 million of their own company's stock in the last 30 days?

Large open-market buys are the insider signal most studies find worth watching.

```shell
curl -X POST https://www.quantisnow.com/api/v1/insider-trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:4 AND transactionCodes:P AND purchased.valueUsd:>1000000 AND filedAt:[2026-10-25 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"purchased.valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 136):

```json
{
  "total": {
    "value": 136,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001011438-26-000643:0001307631",
      "accessionNo": "0001011438-26-000643",
      "formType": "4",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-03T16:35:59-05:00",
      "transactionDate": "2026-11-02",
      "issuer": {
        "cik": "0000789570",
        "name": "MGM Resorts International",
        "ticker": "MGM"
      },
      "reportingOwner": {
        "cik": "0001307631",
        "name": "Meister Keith A.",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [
        "P"
      ],
      "purchased": {
        "shares": 1745592,
        "valueUsd": 51740827.48
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/789570/000101143826000643/0001011438-26-000643-index.htm"
    },
    {
      "id": "0001104659-26-121503:0001820515",
      "accessionNo": "0001104659-26-121503",
      "formType": "4/A",
      "amendment": true,
      "restatesEarlier": false,
      "filedAt": "2026-11-02T17:47:09-05:00",
      "transactionDate": "2026-10-29",
      "issuer": {
        "cik": "0000084839",
        "name": "Rollins Inc",
        "ticker": "ROL"
      },
      "reportingOwner": {
        "cik": "0001820515",
        "name": "R. Randall Rollins Voting Trust U/A dated August 25, 1994",
        "officerTitle": null,
        "isDirector": false,
        "isOfficer": false,
        "isTenPercentOwner": true
      },
      "transactionCodes": [
        "P"
      ],
      "purchased": {
        "shares": 1444402,
        "valueUsd": 49956665.33
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/84839/000110465926121503/0001104659-26-121503-index.htm"
    }
  ]
}
```

## In Python

In [ ]:
import os, requests

URL = "https://www.quantisnow.com/api/v1/insider-trades"
H = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}

def search(q, size=50):
    return requests.post(URL, headers=H, json={"query": q, "size": size}).json()

buys = search("formType:4 AND transactionCodes:P AND purchased.valueUsd:>100000 AND filedAt:[2026-11-16 TO 2026-11-23]")["data"]
for b in buys:
    who, cik = b["reportingOwner"]["cik"], b["issuer"]["cik"]
    earlier = search(f"reportingOwner.cik:{who} AND issuer.cik:{cik} AND transactionCodes:P AND filedAt:<{b['filedAt'][:10]}", size=1)
    if earlier["total"]["value"] == 0:
        print("first buy:", b["issuer"]["ticker"], b["reportingOwner"]["name"], round(b["purchased"]["valueUsd"]))

Each buyer costs one extra call. Our insider records reach back several years, so "first" means first in that history.

## From the command line

```shell
npx quantisnow insiders --min 100k --days 7
npx quantisnow query insider-trades 'reportingOwner.cik:0001402348 AND transactionCodes:P'
```

Related: [Insider buying clusters](https://www.quantisnow.com/sec-api/guides/insider-buying-clusters), [Form 4 API](https://www.quantisnow.com/sec-api/guides/form-4-insider-trading-api).